# Verifying Amplitudes and Correlation Energies

TODO:
- Create a pandas dataframe with the following structure:

structure | basis set | PySCF_Corr | Psi4_Corr | PySCF_norm_t1 | Psi4_norm_t1 | PySCF_norm_t2 | Psi4_norm_t2

- Populate the data frame with the provided code
- Use a try-except to handle linalg error, save a list of faulty structures
- Calculate the mean error for each structure basis combo


In [40]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm


from glob import glob
import psi4
from helper_CC_ML_spacial import *

import pyscf
import pyscf.cc
import pyscf.mcscf


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from numpy.linalg import LinAlgError
import psi4
import pyscf
from pyscf import gto, scf, cc

# Import HelperCCEnergy for Psi4

# Path to XYZ files
xyz_folder = 'diatomics'
xyz_files = glob.glob(os.path.join(xyz_folder, '*.xyz'))

# Extract structure names and contents
structures = []
structure_contents = {}

for file_path in xyz_files:
    structure_name = os.path.splitext(os.path.basename(file_path))[0]
    structures.append(structure_name)
    with open(file_path, 'r') as f:
        structure_contents[structure_name] = f.read()

# Basis sets to iterate over

basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

results = []
failures = []

for structure in structures:
    xyz_path = os.path.join(xyz_folder, f'{structure}.xyz')
    with open(xyz_path, 'r') as f:
        xyz_text = f.read()

    for basis in basis_sets:
        # Initialize default values
        psi4_corr = np.nan
        pyscf_corr = np.nan
        norm_t1_psi4 = np.nan
        norm_t2_psi4 = np.nan
        norm_t1_pyscf = np.nan
        norm_t2_pyscf = np.nan

        ##### PSI4 Processing #####
        try:
            qmol = psi4.qcdb.Molecule.from_string(xyz_text, dtype='xyz')
            mol_psi4 = psi4.geometry(qmol.create_psi4_string_from_molecule() + 'symmetry c1')

            psi4.core.clean()
            psi4.core.be_quiet()

            psi4.set_options({
                'basis': basis,
                'scf_type': 'pk',
                'reference': 'rhf',
                'mp2_type': 'conv',
                'e_convergence': 1e-8,
                'd_convergence': 1e-8
            })

            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            A = HelperCCEnergy(mol_psi4, rhf_e, scf_wfn, freeze_core=False)

            try:
                A.compute_energy()
                psi4_corr = A.FinalEnergy
            except LinAlgError:
                failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Corr'})

            # Extract amplitudes
            norm_t1_psi4 = np.linalg.norm(A.t1)
            norm_t2_psi4 = np.linalg.norm(A.t1)

        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Setup', 'error': str(e)})

        ##### PYSCF Processing #####
        try:
            mol_pyscf = gto.Mole()
            mol_pyscf.build(atom=xyz_path, basis=basis, symmetry='c1')

            # Define active space (assuming no frozen orbitals for simplicity)
            n_frozen = 0
            active_space = range(n_frozen, mol_pyscf.nao_nr())
            scf_calc = scf.RHF(mol_pyscf).run()
            

            # Compute CCSD
            ccsd_calc = cc.CCSD(scf_calc, frozen=[i for i in range(mol_pyscf.nao_nr()) if i not in active_space]).run()
            pyscf_corr = ccsd_calc.e_corr

            t1_pyscf = ccsd_calc.t1
            t2_pyscf = ccsd_calc.t2
            norm_t1_pyscf = np.linalg.norm(t1_pyscf)
            norm_t2_pyscf = np.linalg.norm(t2_pyscf)

        except LinAlgError:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Corr'})
        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Setup', 'error': str(e)})

        # Store results
        results.append({
            'structure': structure,
            'basis set': basis,
            'PySCF_Corr': pyscf_corr,
            'Psi4_Corr': psi4_corr,
            'PySCF_norm_t1': norm_t1_pyscf,
            'Psi4_norm_t1': norm_t1_psi4,
            'PySCF_norm_t2': norm_t2_pyscf,
            'Psi4_norm_t2': norm_t2_psi4
        })

# Create and display final DataFrame
df = pd.DataFrame(results)
print("\nFinal DataFrame:")
print(df)

if failures:
    print("\nFailures Encountered:")
    for failure in failures:
        print(f"Structure: {failure['structure']}, Basis Set: {failure['basis set']}, Method: {failure['method']}, Error: {failure.get('error', '')}")

# Optional: Save results to CSV
# df.to_csv('amplitude_correlation_results.csv', index=False)


SCF not converged.
SCF energy = -90.9792358738535 after 50 cycles  <S^2> = 0.8939958  2S+1 = 2.1391548
E(UCCSD) = -91.16283405922437  E_corr = -0.1835981853709063
converged SCF energy = -92.2129309060755  <S^2> = 1.1444865  2S+1 = 2.3617676
E(UCCSD) = -92.48019073603849  E_corr = -0.267259829963012
converged SCF energy = -92.2170068220425  <S^2> = 1.1429958  2S+1 = 2.3605048
E(UCCSD) = -92.49261506660405  E_corr = -0.2756082445615418
converged SCF energy = -54.1350442502145
E(CCSD) = -54.19702330469032  E_corr = -0.06197905447579966
converged SCF energy = -54.8574063406849
E(CCSD) = -55.01111753825426  E_corr = -0.1537111975693157
converged SCF energy = -54.8653658017596
E(CCSD) = -55.02713553082823  E_corr = -0.1617697290686439
converged SCF energy = -98.1736805380914  <S^2> = 0.95195604  2S+1 = 2.1926751
E(UCCSD) = -98.29054310588637  E_corr = -0.1168625677949854
converged SCF energy = -99.5262384266024  <S^2> = 0.80709928  2S+1 = 2.0563067
E(UCCSD) = -99.77494603190685  E_corr = -0.

In [42]:
failures_df =pd.DataFrame(failures)

In [44]:
df

,structure,basis set,PySCF_Corr,Psi4_Corr,PySCF_norm_t1,Psi4_norm_t1,PySCF_norm_t2,Psi4_norm_t2
0,CN,STO-3G,-0.183598,NaN,NaN,NaN,NaN,NaN
1,CN,cc-pVDZ,-0.267260,NaN,NaN,NaN,NaN,NaN
2,CN,aug-cc-pVDZ,-0.275608,NaN,NaN,NaN,NaN,NaN
3,HN,STO-3G,-0.061979,NaN,0.008507,NaN,1.006604,NaN
4,HN,cc-pVDZ,-0.153711,NaN,0.015478,NaN,0.426481,NaN
...,...,...,...,...,...,...,...,...
103,HO,cc-pVDZ,-0.165945,NaN,NaN,NaN,NaN,NaN
104,HO,aug-cc-pVDZ,-0.179409,NaN,NaN,NaN,NaN,NaN
105,BeH,STO-3G,-0.021963,NaN,NaN,NaN,NaN,NaN
106,BeH,cc-pVDZ,-0.038862,NaN,NaN,NaN,NaN,NaN


In [32]:
df.to_csv('results.csv')


In [30]:
df.loc[df['structure'] == 'BB']

,structure,basis set,PySCF_Corr,Psi4_Corr,PySCF_norm_t1,Psi4_norm_t1,PySCF_norm_t2,Psi4_norm_t2
66,BB,STO-3G,-0.183195,-0.183195,0.124924,0.124924,0.582370,0.124924
67,BB,cc-pVDZ,-0.193752,-0.193752,0.096490,0.096490,0.516084,0.096490
68,BB,aug-cc-pVDZ,-0.195948,-0.195948,0.098257,0.098256,0.517198,0.098256
